<a href="https://colab.research.google.com/github/teklala/ML-Assignment4--Facial-Recognition/blob/main/facial_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
%cd /content/drive/MyDrive/ML-Assignment4--Facial-Recognition/

from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')


Mounted at /content/drive
/content/drive/MyDrive/ML-Assignment4--Facial-Recognition


In [ ]:
import pandas as pd

df = pd.read_csv('./data/fer2013/fer2013.csv')
print(df.head())
print(df['Usage'].value_counts())

   emotion                                             pixels     Usage
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...  Training
1        0  151 150 147 155 148 133 111 140 170 174 182 15...  Training
2        2  231 212 156 164 174 138 161 173 182 200 106 38...  Training
3        4  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...  Training
4        6  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...  Training
Usage
Training       28709
PublicTest      3589
PrivateTest     3589
Name: count, dtype: int64


In [ ]:
# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np

class FER2013Dataset(Dataset):
    def __init__(self, dataframe, split='Training', transform=None):
        self.data = dataframe[dataframe['Usage'] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pixels = self.data.iloc[idx]['pixels']
        image = np.fromstring(pixels, sep=' ', dtype=np.float32).reshape(48, 48)
        image = image / 255.0

        label = int(self.data.iloc[idx]['emotion'])

        if self.transform:
            image = self.transform(image)
        else:
            image = torch.tensor(image).unsqueeze(0)

        return image, torch.tensor(label, dtype=torch.long)

basic_transform = transforms.Compose([transforms.ToTensor()])

train_dataset = FER2013Dataset(df, split='Training', transform=basic_transform)
val_dataset = FER2013Dataset(df, split='PublicTest', transform=basic_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Loaded {len(train_dataset)} training images and {len(val_dataset)} validation images.")

Loaded 28709 training images and 3589 validation images.


In [ ]:
import wandb

wandb.login(key=userdata.get('WANDB_API_KEY'))
print("Weights & Biases connected")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tvada22 (tvada22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Weights & Biases connected


In [ ]:
# Architecture 1 - Baseline MLP
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

class BaselineMLP(nn.Module):
    def __init__(self):
        super(BaselineMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(48 * 48, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 7)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

Compute Device: cpu


In [ ]:
def train_model(model_class, config, architecture_name):
    run_name = f"lr_{config['learning_rate']}_bs_{config['batch_size']}"
    run = wandb.init(
        project="Facial-Recognition-Challenge",
        group=architecture_name,
        name=run_name,
        config=config
    )

    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()

    wd = config.get('weight_decay', 0.0)

    if config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=wd)
    else:
        optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.9, weight_decay=wd)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    print(f"Starting Training: {architecture_name}")

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)

        model.eval()
        correct = 0
        total = 0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct / total

        print(f"Epoch [{epoch}/{config['epochs']}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

        scheduler.step(avg_val_loss)
        wandb.log({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_accuracy
        })


    model_path = f'./models/{architecture_name}_{run_name}.pth'
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")

    run.finish()
    print("Run Complete")
    return model

In [ ]:
config_baseline = {
    "learning_rate": 0.01,
    "batch_size": 64,
    "epochs": 10,
    "optimizer": "SGD"
}

baseline_model = train_model(
    model_class=BaselineMLP,
    config=config_baseline,
    architecture_name="Architecture_1_Baseline_MLP"
)

Starting Training: Architecture_1_Baseline_MLP
Epoch [1/10] - Train Loss: 1.7562 | Val Loss: 1.7420 | Val Acc: 0.2923
Epoch [2/10] - Train Loss: 1.6859 | Val Loss: 1.6521 | Val Acc: 0.3566
Epoch [3/10] - Train Loss: 1.6697 | Val Loss: 1.6465 | Val Acc: 0.3644
Epoch [4/10] - Train Loss: 1.6506 | Val Loss: 1.6464 | Val Acc: 0.3461
Epoch [5/10] - Train Loss: 1.6391 | Val Loss: 1.6119 | Val Acc: 0.3792
Epoch [6/10] - Train Loss: 1.6256 | Val Loss: 1.6171 | Val Acc: 0.3650
Epoch [7/10] - Train Loss: 1.6178 | Val Loss: 1.6138 | Val Acc: 0.3784
Epoch [8/10] - Train Loss: 1.5995 | Val Loss: 1.6047 | Val Acc: 0.3745
Epoch [9/10] - Train Loss: 1.5930 | Val Loss: 1.6235 | Val Acc: 0.3731
Epoch [10/10] - Train Loss: 1.5917 | Val Loss: 1.5682 | Val Acc: 0.3848
Model saved to ./models/Architecture_1_Baseline_MLP_lr_0.01_bs_64.pth


epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▄▄▃▂▂▁▁▁
val_accuracy,▁▆▆▅█▇█▇▇█
val_loss,█▄▄▄▃▃▃▂▃▁
epoch,10
train_loss,1.59171
val_accuracy,0.38479
val_loss,1.56825


Run Complete


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

class OverfitterCNN(nn.Module):
    def __init__(self):
        super(OverfitterCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(128 * 6 * 6, 512)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(512, 7)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.flatten(x)
        x = self.relu4(self.fc1(x))
        x = self.fc2(x)
        return x

Compute Device: cuda


In [ ]:
# (FORWARD & BACKWARD PASS)
import torch
import torch.nn as nn

test_model =  OverfitterCNN().to(device)
test_criterion = nn.CrossEntropyLoss()

dummy_images = torch.randn(16, 1, 48, 48).to(device)
dummy_labels = torch.randint(0, 7, (16,)).to(device)

outputs = test_model(dummy_images)
print(f" Forward Pass Success. Expected shape [16, 7], got {outputs.shape}")

loss = test_criterion(outputs, dummy_labels)
loss.backward()

has_gradients = test_model.conv1.weight.grad is not None
print(f" Backward Pass Success. Gradients computed: {has_gradients}")

 Forward Pass Success. Expected shape [16, 7], got torch.Size([16, 7])
 Backward Pass Success. Gradients computed: True


In [ ]:
#single batch test
import torch
import torch.nn as nn
import torch.optim as optim

model = OverfitterCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

print(f"Targeting a single batch of shape: {images.shape}")
print("Loss should approach 0.0, Accuracy should approach 1.0\n")

for step in range(1, 101):
    model.train()

    # Forward pass
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Backward pass
    loss.backward()
    optimizer.step()

    _, predicted = torch.max(outputs.data, 1)
    total = labels.size(0)
    correct = (predicted == labels).sum().item()
    accuracy = correct / total

    if step == 1 or step % 10 == 0:
        print(f"Iteration [{step:3d}/100] -> Loss: {loss.item():.4f} | Accuracy: {accuracy:.4f}")

    if loss.item() < 1e-4 and accuracy == 1.0:
        print(f"\n Perfect memorization achieved early at iteration {step}!")
        break

Targeting a single batch of shape: torch.Size([64, 1, 48, 48])
Loss should approach 0.0, Accuracy should approach 1.0

Iteration [  1/100] -> Loss: 1.9456 | Accuracy: 0.1719
Iteration [ 10/100] -> Loss: 1.6192 | Accuracy: 0.4531
Iteration [ 20/100] -> Loss: 0.9710 | Accuracy: 0.6406
Iteration [ 30/100] -> Loss: 0.3762 | Accuracy: 0.8750
Iteration [ 40/100] -> Loss: 0.0490 | Accuracy: 1.0000
Iteration [ 50/100] -> Loss: 0.0031 | Accuracy: 1.0000
Iteration [ 60/100] -> Loss: 0.0005 | Accuracy: 1.0000
Iteration [ 70/100] -> Loss: 0.0002 | Accuracy: 1.0000
Iteration [ 80/100] -> Loss: 0.0001 | Accuracy: 1.0000

 Perfect memorization achieved early at iteration 88!


In [ ]:
config_cnn_overfit = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 15,
    "optimizer": "Adam"
}

overfitter_model = train_model(
    model_class=OverfitterCNN,
    config=config_cnn_overfit,
    architecture_name="Architecture_2_Overfitter_CNN"
)

Starting Training: Architecture_2_Overfitter_CNN
Epoch [1/15] - Train Loss: 1.6777 | Val Loss: 1.5154 | Val Acc: 0.4230
Epoch [2/15] - Train Loss: 1.4452 | Val Loss: 1.3631 | Val Acc: 0.4759
Epoch [3/15] - Train Loss: 1.2934 | Val Loss: 1.2766 | Val Acc: 0.5099
Epoch [4/15] - Train Loss: 1.1701 | Val Loss: 1.2274 | Val Acc: 0.5380
Epoch [5/15] - Train Loss: 1.0616 | Val Loss: 1.2693 | Val Acc: 0.5171
Epoch [6/15] - Train Loss: 0.9488 | Val Loss: 1.2245 | Val Acc: 0.5408
Epoch [7/15] - Train Loss: 0.8211 | Val Loss: 1.2384 | Val Acc: 0.5637
Epoch [8/15] - Train Loss: 0.6764 | Val Loss: 1.3378 | Val Acc: 0.5564
Epoch [9/15] - Train Loss: 0.5373 | Val Loss: 1.4621 | Val Acc: 0.5500
Epoch [10/15] - Train Loss: 0.3963 | Val Loss: 1.7061 | Val Acc: 0.5626
Epoch [11/15] - Train Loss: 0.2761 | Val Loss: 1.9125 | Val Acc: 0.5553
Epoch [12/15] - Train Loss: 0.1977 | Val Loss: 2.2579 | Val Acc: 0.5508
Epoch [13/15] - Train Loss: 0.1367 | Val Loss: 2.5490 | Val Acc: 0.5506
Epoch [14/15] - Train Lo

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▇▆▆▅▅▄▄▃▂▂▁▁▁▁
val_accuracy,▁▄▅▇▆▇██▇██▇▇▇▇
val_loss,▂▂▁▁▁▁▁▁▂▃▄▅▇▇█
epoch,15
train_loss,0.09737
val_accuracy,0.54946
val_loss,2.8737


Run Complete


In [ ]:
# Architecture 3
import torchvision.transforms as transforms
# data augmentation
train_transform_aug = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
])

train_dataset_aug = FER2013Dataset(df, split='Training', transform=train_transform_aug)

train_loader = DataLoader(train_dataset_aug, batch_size=64, shuffle=True)


In [ ]:
class OptimalCNN(nn.Module):
    def __init__(self):
        super(OptimalCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(128 * 6 * 6, 512)
        self.drop1 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(512, 7)

    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.drop1(x)
        x = self.fc2(x)
        return x

In [ ]:
config_cnn_optimal = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 20,
    "optimizer": "Adam"
}

optimal_model = train_model(
    model_class=OptimalCNN,
    config=config_cnn_optimal,
    architecture_name="Architecture_3_Optimal_CNN"
)

Starting Training: Architecture_3_Optimal_CNN
Epoch [1/20] - Train Loss: 1.7822 | Val Loss: 1.6080 | Val Acc: 0.3697
Epoch [2/20] - Train Loss: 1.5565 | Val Loss: 1.3920 | Val Acc: 0.4620
Epoch [3/20] - Train Loss: 1.4770 | Val Loss: 1.4193 | Val Acc: 0.4486
Epoch [4/20] - Train Loss: 1.4395 | Val Loss: 1.3698 | Val Acc: 0.4784
Epoch [5/20] - Train Loss: 1.3958 | Val Loss: 1.2761 | Val Acc: 0.5149
Epoch [6/20] - Train Loss: 1.3697 | Val Loss: 1.2629 | Val Acc: 0.5138
Epoch [7/20] - Train Loss: 1.3472 | Val Loss: 1.2565 | Val Acc: 0.5155
Epoch [8/20] - Train Loss: 1.3210 | Val Loss: 1.2129 | Val Acc: 0.5430
Epoch [9/20] - Train Loss: 1.3010 | Val Loss: 1.1949 | Val Acc: 0.5313
Epoch [10/20] - Train Loss: 1.2813 | Val Loss: 1.2065 | Val Acc: 0.5364
Epoch [11/20] - Train Loss: 1.2693 | Val Loss: 1.1737 | Val Acc: 0.5545
Epoch [12/20] - Train Loss: 1.2503 | Val Loss: 1.1549 | Val Acc: 0.5587
Epoch [13/20] - Train Loss: 1.2357 | Val Loss: 1.1376 | Val Acc: 0.5609
Epoch [14/20] - Train Loss:

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▅▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val_accuracy,▁▄▄▅▆▆▆▇▇▇▇▇█▇▇█████
val_loss,█▅▅▅▃▃▃▃▂▃▂▂▂▂▂▁▁▁▁▁
epoch,20
train_loss,1.1618
val_accuracy,0.57481
val_loss,1.095


Run Complete


In [ ]:
# Architecture 4

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")


class VGGStyleCNN(nn.Module):
    def __init__(self):
        super(VGGStyleCNN, self).__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.flatten = nn.Flatten()

        self.classifier = nn.Sequential(
            nn.Linear(256 * 6 * 6, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

Compute Device: cuda


In [ ]:
config_vgg_style = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 30,
    "optimizer": "Adam"
}

vgg_model = train_model(
    model_class=VGGStyleCNN,
    config=config_vgg_style,
    architecture_name="Architecture_4_VGG_Style_CNN"
)

Starting Training: Architecture_4_VGG_Style_CNN
Epoch [1/30] - Train Loss: 1.4405 | Val Loss: 1.2678 | Val Acc: 0.5169
Epoch [2/30] - Train Loss: 1.1639 | Val Loss: 1.1869 | Val Acc: 0.5561
Epoch [3/30] - Train Loss: 1.0436 | Val Loss: 1.0986 | Val Acc: 0.5809
Epoch [4/30] - Train Loss: 0.9457 | Val Loss: 1.0693 | Val Acc: 0.6046
Epoch [5/30] - Train Loss: 0.8507 | Val Loss: 1.0499 | Val Acc: 0.6060
Epoch [6/30] - Train Loss: 0.7443 | Val Loss: 1.0889 | Val Acc: 0.6163
Epoch [7/30] - Train Loss: 0.6186 | Val Loss: 1.1370 | Val Acc: 0.6264
Epoch [8/30] - Train Loss: 0.4751 | Val Loss: 1.2084 | Val Acc: 0.6353
Epoch [9/30] - Train Loss: 0.3478 | Val Loss: 1.3095 | Val Acc: 0.6241
Epoch [10/30] - Train Loss: 0.1605 | Val Loss: 1.3601 | Val Acc: 0.6456
Epoch [11/30] - Train Loss: 0.0887 | Val Loss: 1.4923 | Val Acc: 0.6378
Epoch [12/30] - Train Loss: 0.0684 | Val Loss: 1.6156 | Val Acc: 0.6367
Epoch [13/30] - Train Loss: 0.0617 | Val Loss: 1.7254 | Val Acc: 0.6406
Epoch [14/30] - Train Los

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▇▆▆▅▅▄▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▃▄▆▆▆▇▇▇█▇▇██████████████████
val_loss,▂▂▁▁▁▁▂▂▃▃▄▅▆▆▆▇▇▇▇▇▇▇▇███████
epoch,30
train_loss,0.00766
val_accuracy,0.64363
val_loss,2.08051


Run Complete


In [ ]:
print(" Lower LR, L2 Regularization")

config_vgg_tuned = {
    "learning_rate": 0.0005,
    "batch_size": 64,
    "epochs": 30,
    "optimizer": "Adam",
    "weight_decay": 1e-3
}

vgg_tuned = train_model(
    model_class=VGGStyleCNN,
    config=config_vgg_tuned,
    architecture_name="Architecture_4_VGG_Tuned"
)

 Lower LR, L2 Regularization


Starting Training: Architecture_4_VGG_Tuned
Epoch [1/30] - Train Loss: 1.4128 | Val Loss: 1.2432 | Val Acc: 0.5127
Epoch [2/30] - Train Loss: 1.1473 | Val Loss: 1.2030 | Val Acc: 0.5430
Epoch [3/30] - Train Loss: 1.0511 | Val Loss: 1.0760 | Val Acc: 0.5913
Epoch [4/30] - Train Loss: 0.9843 | Val Loss: 1.0782 | Val Acc: 0.5977
Epoch [5/30] - Train Loss: 0.9250 | Val Loss: 1.0588 | Val Acc: 0.6049
Epoch [6/30] - Train Loss: 0.8679 | Val Loss: 1.0694 | Val Acc: 0.5971
Epoch [7/30] - Train Loss: 0.8002 | Val Loss: 1.0762 | Val Acc: 0.5960
Epoch [8/30] - Train Loss: 0.7334 | Val Loss: 1.0825 | Val Acc: 0.6116
Epoch [9/30] - Train Loss: 0.6386 | Val Loss: 1.1337 | Val Acc: 0.6052
Epoch [10/30] - Train Loss: 0.3612 | Val Loss: 1.2430 | Val Acc: 0.6230
Epoch [11/30] - Train Loss: 0.2038 | Val Loss: 1.3995 | Val Acc: 0.6018
Epoch [12/30] - Train Loss: 0.1546 | Val Loss: 1.4066 | Val Acc: 0.6119
Epoch [13/30] - Train Loss: 0.1313 | Val Loss: 1.5177 | Val Acc: 0.6057
Epoch [14/30] - Train Loss: 0

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▇▆▆▆▅▅▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▃▆▆▆▆▆▇▆▇▆▇▆██████▇▇█████████
val_loss,▃▃▁▁▁▁▁▁▂▃▅▅▇▆▆▆▇▇▇█▇▇▇▇▇▇▇▇▇▇
epoch,30
train_loss,0.00704
val_accuracy,0.62914
val_loss,1.52114


Run Complete
